In [1]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ==================== 配置参数 ====================
# 请根据您的实际文件路径修改
dir = '/Users/dingyu_xuan/Desktop/Project_IC_CSU/OSM/'
SHAPEFILE_PATH = dir + "building_selected_buffer.shp"  # 多边形Shapefile路径
PNG_PATH = dir + "satellite_IC.png"                     # 卫星影像PNG路径
OUTPUT_DIR = dir + "output_masks"                       # 输出文件夹名称
NAME_FIELD = "name"                               # 名称字段名
ID_FIELD = "osm_id"                               # 备用ID字段名

# 创建输出目录
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ==================== 读取数据 ====================
print("正在读取Shapefile...")
gdf = gpd.read_file(SHAPEFILE_PATH)
print(f"共找到 {len(gdf)} 个多边形")

# 确保坐标系是WGS84（经纬度），rasterio读取PNG时需要对齐
# 如果PNG没有地理信息，我们只按像素位置裁剪
print("正在读取PNG图像...")
try:
    # 尝试作为带地理坐标的GeoTIFF读取
    src = rasterio.open(PNG_PATH)
    has_georef = True
    print("PNG文件包含地理参考信息")
except:
    # 如果是普通PNG，使用matplotlib读取
    has_georef = False
    img = plt.imread(PNG_PATH)
    print("PNG文件为普通图像，将按像素位置裁剪（需要手动对齐）")

# ==================== 遍历每个多边形并导出 ====================
for idx, row in gdf.iterrows():
    # 获取文件名：优先使用name，为空则用osm_id
    name_val = row.get(NAME_FIELD, "")
    osm_id_val = row.get(ID_FIELD, "")
    
    if name_val and str(name_val) != "" and str(name_val) != "nan":
        filename = str(name_val).replace("/", "_").replace("\\", "_")  # 去除非法字符
    else:
        filename = f"id_{osm_id_val}" if osm_id_val and str(osm_id_val) != "nan" else f"polygon_{idx}"
    
    output_path = Path(OUTPUT_DIR) / f"{filename}.png"
    
    print(f"正在处理 [{idx+1}/{len(gdf)}]: {filename}")
    
    try:
        if has_georef:
            # ===== 方法1：带地理坐标的影像 =====
            geom = row.geometry
            # 将多边形转换为GeoJSON格式（rasterio需要）
            geoms = [mapping(geom)]
            # 裁剪影像
            out_image, out_transform = mask(src, geoms, crop=True)
            # 移除alpha通道（如果有）
            out_image = out_image[:3]  # 只保留RGB三通道
            # 转置为适合matplotlib保存的格式 (rows, cols, bands)
            out_image = np.transpose(out_image, (1, 2, 0))
            # 保存为PNG
            plt.imsave(output_path, out_image)
        else:
            # ===== 方法2：普通PNG（无地理坐标） =====
            # 注意：这种方法需要多边形与图像的像素坐标对齐
            # 这里假设多边形已经是像素坐标系下的范围
            geom = row.geometry
            # 获取多边形边界（像素坐标）
            minx, miny, maxx, maxy = geom.bounds
            # 转换为整数索引
            x1, y1, x2, y2 = int(minx), int(miny), int(maxx), int(maxy)
            # 确保范围在图像内
            h, w = img.shape[:2]
            x1, x2 = max(0, x1), min(w, x2)
            y1, y2 = max(0, y1), min(h, y2)
            # 裁剪并保存
            cropped = img[y1:y2, x1:x2]
            plt.imsave(output_path, cropped)
            
        print(f"    ✓ 已保存: {output_path}")
        
    except Exception as e:
        print(f"    ✗ 处理失败: {e}")

# 关闭文件
if has_georef:
    src.close()

print(f"\n完成！共导出 {len(gdf)} 个文件，保存在 '{OUTPUT_DIR}' 文件夹中")

正在读取Shapefile...
共找到 76 个多边形
正在读取PNG图像...
PNG文件包含地理参考信息
正在处理 [1/76]: Blackett Laboratory
    ✓ 已保存: /Users/dingyu_xuan/Desktop/Project_IC_CSU/OSM/output_masks/Blackett Laboratory.png
正在处理 [2/76]: Royal Geographical Society
    ✓ 已保存: /Users/dingyu_xuan/Desktop/Project_IC_CSU/OSM/output_masks/Royal Geographical Society.png
正在处理 [3/76]: Royal College of Music
    ✓ 已保存: /Users/dingyu_xuan/Desktop/Project_IC_CSU/OSM/output_masks/Royal College of Music.png
正在处理 [4/76]: Holy Trinity
    ✓ 已保存: /Users/dingyu_xuan/Desktop/Project_IC_CSU/OSM/output_masks/Holy Trinity.png
正在处理 [5/76]: id_27917475
    ✓ 已保存: /Users/dingyu_xuan/Desktop/Project_IC_CSU/OSM/output_masks/id_27917475.png
正在处理 [6/76]: Albert Hall Mansions (49-86)
    ✓ 已保存: /Users/dingyu_xuan/Desktop/Project_IC_CSU/OSM/output_masks/Albert Hall Mansions (49-86).png
正在处理 [7/76]: Roderic Hill Building
    ✓ 已保存: /Users/dingyu_xuan/Desktop/Project_IC_CSU/OSM/output_masks/Roderic Hill Building.png
正在处理 [8/76]: Rector's Residence
    ✓ 已保存: 